In [1]:
import requests
import pandas as pd
import os
import re

C:\Users\10475\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\10475\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
C:\Users\10475\AppData\Local\Temp\ipykernel_25860\903929269.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
url = 'https://materialsdb.cn/2dsdb/index.html?'
headers = {
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36 Edg/111.0.1661.51'
}
response = requests.get(url=url,headers=headers)
print(response)

<Response [200]>


In [3]:
html_data = re.findall('<li><p><a class="reference external" href="(.*?).html">(.*?)</a></p></li>',response.text)
print(len(html_data))

274


In [4]:
filename = 'materials\\'
if not os.path.exists(filename):
    os.mkdir(filename)

In [5]:
materials_id = []
materials_num = re.sub(r'\^','/',str(html_data[0][0]))
num = re.sub(r'\^','%5E',str(html_data[0][0]))

In [6]:
materials_id = []
for num_id in html_data:
    materials_num = re.sub(r'\^','/',str(num_id[0]))
    materials_id.append(materials_num)
    num = re.sub(r'\^','%5E',str(num_id[0]))
    materials_url = f'https://materialsdb.cn/2dsdb/{num}.html'
    print(materials_url)

https://materialsdb.cn/2dsdb/As_P21-m.html
https://materialsdb.cn/2dsdb/As_P-3m1.html
https://materialsdb.cn/2dsdb/As_Pca21.html
https://materialsdb.cn/2dsdb/As_Pmn2_1.html
https://materialsdb.cn/2dsdb/As_Pmna.html
https://materialsdb.cn/2dsdb/C_P6%5Emmm.html
https://materialsdb.cn/2dsdb/Ge_P-3m1.html
https://materialsdb.cn/2dsdb/P_P21-m.html
https://materialsdb.cn/2dsdb/P_Pca21.html
https://materialsdb.cn/2dsdb/P_Pmna.html
https://materialsdb.cn/2dsdb/Si_P-3m1.html
https://materialsdb.cn/2dsdb/AlN_P-6m2.html
https://materialsdb.cn/2dsdb/AuSe_P2%5Em.html
https://materialsdb.cn/2dsdb/BN_P-6m2.html
https://materialsdb.cn/2dsdb/GaN_P-6m2.html
https://materialsdb.cn/2dsdb/GaSe_P-6m2.html
https://materialsdb.cn/2dsdb/GaS_P-3m1.html
https://materialsdb.cn/2dsdb/GaS_P-6m2.html
https://materialsdb.cn/2dsdb/GeAs_P-6m2.html
https://materialsdb.cn/2dsdb/GeP_P-6m2.html
https://materialsdb.cn/2dsdb/GeSe_Pmn2_1.html
https://materialsdb.cn/2dsdb/InN_P-6m2.html
https://materialsdb.cn/2dsdb/InSe_P3m1.h

In [7]:
#获取dos图
filename = 'graph\\'
if not os.path.exists(filename):
    os.mkdir(filename)

In [ ]:
for num_id in html_data:
    graph_url = f'https://materialsdb.cn/_images/BAND_LDOS-{num_id[0]}.png'
    graph_content = requests.get(url=graph_url,headers=headers).content
    with open('graph\\' + num_id[0] + '.png',mode='wb') as f:
        f.write(graph_content)

In [ ]:
#获取空间群
materials_id = []
space_group = []
for num_id in html_data:
    materials_num = re.sub(r'\^','/',str(num_id[0]))
    materials_id.append(materials_num)
    num = re.sub(r'\^','%5E',str(num_id[0]))
    materials_url = f'https://materialsdb.cn/2dsdb/{num}.html'
    response = requests.get(url=materials_url,headers=headers)
    soup = BeautifulSoup(response.content,'html.parser')
    paragraphs = [p.text for p in soup.find_all('p')]
    space_group.append(paragraphs[12])
    print(space_group)

In [ ]:
materials_space_group = pd.DataFrame({"name":materials_id,"space_group":space_group})
materials_space_group.to_csv("materials_space_group.csv",index=False)

# matminer自动提取材料特征

In [8]:
from pymatgen.ext.matproj import MPRester #短API
#from mp_api.client import MPRester #长API

In [12]:
with MPRester("DxTbWWBsxEID35nq")as mpr:
    criteria = {"elements":{"$all":['O','Li','Fe']},"nelements":3}
    property = ["material_id","pretty_formula","energy_per_atom","spacegroup","structure","density"]
    data = (mpr.query(criteria=criteria,properties=property))
    print(data)

[{'material_id': 'mp-756155', 'pretty_formula': 'Li6FeO4', 'energy_per_atom': -5.117325523181818, 'spacegroup': {'symprec': 0.1, 'source': 'spglib', 'symbol': 'P4_2/nmc', 'number': 137, 'point_group': 'mmm', 'crystal_system': 'tetragonal', 'hall': 'P 4n 2n -1n'}, 'structure': Structure Summary
Lattice
    abc : 6.686419029417614 8.160871258593042 8.159255157088115
 angles : 110.17051264942266 89.99800706143297 90.01037625621055
 volume : 417.9206613437442
      A : 6.686419 -0.000441 -0.000446
      B : -0.000725 6.691772 4.671189
      C : 0.000154 -6.690807 4.669748
    pbc : True True True
PeriodicSite: Li (1.671, 1.673, 3.261) [0.25, 0.4742, 0.2241]
PeriodicSite: Li (1.671, 1.674, 7.932) [0.25, 0.9742, 0.7241]
PeriodicSite: Li (1.671, -5.018, 5.596) [0.25, 0.2242, 0.9741]
PeriodicSite: Li (1.671, 1.674, 5.597) [0.25, 0.7242, 0.4741]
PeriodicSite: Li (5.015, -1.672, 6.079) [0.75, 0.5259, 0.7758]
PeriodicSite: Li (5.015, -1.673, 1.409) [0.75, 0.02587, 0.2758]
PeriodicSite: Li (5.014,

In [13]:
data = pd.DataFrame(data)
data

,material_id,pretty_formula,energy_per_atom,spacegroup,structure,density
0,mp-756155,Li6FeO4,-5.117326,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.67125537 1.67346391 3.26141012] Li, [1.670...",2.566594
1,mp-756856,Li5Fe7O12,-6.183322,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0.74461757 4.34819314 2.48026726] Li, [-0.73...",4.516352
2,mp-762529,Li2(FeO2)5,-5.740312,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-0.35339594 3.48269285 3.02141686] Li, [ 2...",4.046393
3,mp-18782,LiFeO2,-6.085931,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0. 0. 4.377058] Li, [2.061247 0....",4.231705
4,mp-756497,LiFeO2,-6.078770,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[3.04987747 3.81109335 4.24134412] Li, [3.013...",4.143655
...,...,...,...,...,...,...
155,mp-1101736,LiFe3O4,-6.351436,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-1.48982311 2.19943369 -1.43482144] Li, [-1...",5.018471
156,mp-780216,Li5Fe3O8,-5.304033,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.33531488 -4.11912064 -0.49520662] Li, [-0...",3.757134
157,mp-759394,Li2FeO3,-5.613536,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.244898 0. 4.9203229] Li, [1.244898...",3.651901
158,mp-1314324,LiFe2O3,-6.267500,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.52274007 -2.68996763 3.71993333] Li, [ 3...",4.625861


In [14]:
from matminer.featurizers.structure import XRDPowderPattern

xrd = XRDPowderPattern(two_theta_range=[0,299])
xrd.feature_labels()
data = xrd.featurize_dataframe(data,"structure")
data

XRDPowderPattern:   0%|          | 0/160 [00:00<?, ?it/s]

,material_id,pretty_formula,energy_per_atom,spacegroup,structure,density,xrd_0,xrd_1,xrd_2,xrd_3,...,xrd_290,xrd_291,xrd_292,xrd_293,xrd_294,xrd_295,xrd_296,xrd_297,xrd_298,xrd_299
0,mp-756155,Li6FeO4,-5.117326,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.67125537 1.67346391 3.26141012] Li, [1.670...",2.566594,5.977826e-179,1.379972e-160,3.144998e-143,7.076099e-127,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,mp-756856,Li5Fe7O12,-6.183322,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0.74461757 4.34819314 2.48026726] Li, [-0.73...",4.516352,1.186238e-56,2.163227e-45,1.940292e-35,8.559855e-27,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,mp-762529,Li2(FeO2)5,-5.740312,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-0.35339594 3.48269285 3.02141686] Li, [ 2...",4.046393,1.552619e-78,1.175554e-66,8.898801e-56,6.734917e-46,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,mp-18782,LiFeO2,-6.085931,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0. 0. 4.377058] Li, [2.061247 0....",4.231705,0.000000e+00,0.000000e+00,9.881313e-322,3.397916e-293,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,mp-756497,LiFeO2,-6.078770,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[3.04987747 3.81109335 4.24134412] Li, [3.013...",4.143655,3.678547e-82,1.400721e-69,4.517527e-58,1.234025e-47,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,mp-1101736,LiFe3O4,-6.351436,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-1.48982311 2.19943369 -1.43482144] Li, [-1...",5.018471,2.624011e-228,1.241719e-203,2.209240e-180,1.478050e-158,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
156,mp-780216,Li5Fe3O8,-5.304033,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.33531488 -4.11912064 -0.49520662] Li, [-0...",3.757134,6.358424e-144,2.845674e-128,1.529721e-113,9.882739e-100,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
157,mp-759394,Li2FeO3,-5.613536,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.244898 0. 4.9203229] Li, [1.244898...",3.651901,1.407637e-221,1.260348e-200,1.010579e-180,7.256715e-162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
158,mp-1314324,LiFe2O3,-6.267500,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.52274007 -2.68996763 3.71993333] Li, [ 3...",4.625861,4.635395e-58,4.953273e-49,8.621947e-41,2.444703e-33,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
from matminer.featurizers.conversions import StrToComposition

data = StrToComposition().featurize_dataframe(data,"pretty_formula")
data

StrToComposition:   0%|          | 0/160 [00:00<?, ?it/s]

,material_id,pretty_formula,energy_per_atom,spacegroup,structure,density,xrd_0,xrd_1,xrd_2,xrd_3,...,xrd_291,xrd_292,xrd_293,xrd_294,xrd_295,xrd_296,xrd_297,xrd_298,xrd_299,composition
0,mp-756155,Li6FeO4,-5.117326,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.67125537 1.67346391 3.26141012] Li, [1.670...",2.566594,5.977826e-179,1.379972e-160,3.144998e-143,7.076099e-127,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"(Li, Fe, O)"
1,mp-756856,Li5Fe7O12,-6.183322,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0.74461757 4.34819314 2.48026726] Li, [-0.73...",4.516352,1.186238e-56,2.163227e-45,1.940292e-35,8.559855e-27,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"(Li, Fe, O)"
2,mp-762529,Li2(FeO2)5,-5.740312,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-0.35339594 3.48269285 3.02141686] Li, [ 2...",4.046393,1.552619e-78,1.175554e-66,8.898801e-56,6.734917e-46,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"(Li, Fe, O)"
3,mp-18782,LiFeO2,-6.085931,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0. 0. 4.377058] Li, [2.061247 0....",4.231705,0.000000e+00,0.000000e+00,9.881313e-322,3.397916e-293,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"(Li, Fe, O)"
4,mp-756497,LiFeO2,-6.078770,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[3.04987747 3.81109335 4.24134412] Li, [3.013...",4.143655,3.678547e-82,1.400721e-69,4.517527e-58,1.234025e-47,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"(Li, Fe, O)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,mp-1101736,LiFe3O4,-6.351436,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-1.48982311 2.19943369 -1.43482144] Li, [-1...",5.018471,2.624011e-228,1.241719e-203,2.209240e-180,1.478050e-158,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"(Li, Fe, O)"
156,mp-780216,Li5Fe3O8,-5.304033,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.33531488 -4.11912064 -0.49520662] Li, [-0...",3.757134,6.358424e-144,2.845674e-128,1.529721e-113,9.882739e-100,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"(Li, Fe, O)"
157,mp-759394,Li2FeO3,-5.613536,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.244898 0. 4.9203229] Li, [1.244898...",3.651901,1.407637e-221,1.260348e-200,1.010579e-180,7.256715e-162,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"(Li, Fe, O)"
158,mp-1314324,LiFe2O3,-6.267500,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.52274007 -2.68996763 3.71993333] Li, [ 3...",4.625861,4.635395e-58,4.953273e-49,8.621947e-41,2.444703e-33,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"(Li, Fe, O)"


In [16]:
from matminer.featurizers.composition import ElementProperty

ep_feat = ElementProperty.from_preset(preset_name="magpie")
data = ep_feat.featurize_dataframe(data,col_id="composition")
data

ElementProperty:   0%|          | 0/160 [00:00<?, ?it/s]

,material_id,pretty_formula,energy_per_atom,spacegroup,structure,density,xrd_0,xrd_1,xrd_2,xrd_3,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,mp-756155,Li6FeO4,-5.117326,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.67125537 1.67346391 3.26141012] Li, [1.670...",2.566594,5.977826e-179,1.379972e-160,3.144998e-143,7.076099e-127,...,2.110663,0.191878,0.348870,0.0,12.0,229.0,217.0,150.090909,100.429752,229.0
1,mp-756856,Li5Fe7O12,-6.183322,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0.74461757 4.34819314 2.48026726] Li, [-0.73...",4.516352,1.186238e-56,2.163227e-45,1.940292e-35,8.559855e-27,...,2.110663,0.615610,0.872114,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0
2,mp-762529,Li2(FeO2)5,-5.740312,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-0.35339594 3.48269285 3.02141686] Li, [ 2...",4.046393,1.552619e-78,1.175554e-66,8.898801e-56,6.734917e-46,...,2.110663,0.620783,0.876400,0.0,12.0,229.0,217.0,101.352941,105.121107,12.0
3,mp-18782,LiFeO2,-6.085931,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0. 0. 4.377058] Li, [2.061247 0....",4.231705,0.000000e+00,0.000000e+00,9.881313e-322,3.397916e-293,...,2.110663,0.527666,0.791499,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0
4,mp-756497,LiFeO2,-6.078770,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[3.04987747 3.81109335 4.24134412] Li, [3.013...",4.143655,3.678547e-82,1.400721e-69,4.517527e-58,1.234025e-47,...,2.110663,0.527666,0.791499,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,mp-1101736,LiFe3O4,-6.351436,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-1.48982311 2.19943369 -1.43482144] Li, [-1...",5.018471,2.624011e-228,1.241719e-203,2.209240e-180,1.478050e-158,...,2.110663,0.791499,0.989373,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0
156,mp-780216,Li5Fe3O8,-5.304033,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.33531488 -4.11912064 -0.49520662] Li, [-0...",3.757134,6.358424e-144,2.845674e-128,1.529721e-113,9.882739e-100,...,2.110663,0.395749,0.643093,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0
157,mp-759394,Li2FeO3,-5.613536,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.244898 0. 4.9203229] Li, [1.244898...",3.651901,1.407637e-221,1.260348e-200,1.010579e-180,7.256715e-162,...,2.110663,0.351777,0.586295,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0
158,mp-1314324,LiFe2O3,-6.267500,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.52274007 -2.68996763 3.71993333] Li, [ 3...",4.625861,4.635395e-58,4.953273e-49,8.621947e-41,2.444703e-33,...,2.110663,0.703554,0.938072,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0


In [17]:
from matminer.featurizers.conversions import CompositionToOxidComposition

data = CompositionToOxidComposition().featurize_dataframe(data,"composition",ignore_errors = True)
data

CompositionToOxidComposition:   0%|          | 0/160 [00:00<?, ?it/s]

,material_id,pretty_formula,energy_per_atom,spacegroup,structure,density,xrd_0,xrd_1,xrd_2,xrd_3,...,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber,composition_oxid
0,mp-756155,Li6FeO4,-5.117326,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.67125537 1.67346391 3.26141012] Li, [1.670...",2.566594,5.977826e-179,1.379972e-160,3.144998e-143,7.076099e-127,...,0.191878,0.348870,0.0,12.0,229.0,217.0,150.090909,100.429752,229.0,"(Li+, Fe2+, O2-)"
1,mp-756856,Li5Fe7O12,-6.183322,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0.74461757 4.34819314 2.48026726] Li, [-0.73...",4.516352,1.186238e-56,2.163227e-45,1.940292e-35,8.559855e-27,...,0.615610,0.872114,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)"
2,mp-762529,Li2(FeO2)5,-5.740312,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-0.35339594 3.48269285 3.02141686] Li, [ 2...",4.046393,1.552619e-78,1.175554e-66,8.898801e-56,6.734917e-46,...,0.620783,0.876400,0.0,12.0,229.0,217.0,101.352941,105.121107,12.0,"(Li0+, Fe0+, O0+)"
3,mp-18782,LiFeO2,-6.085931,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0. 0. 4.377058] Li, [2.061247 0....",4.231705,0.000000e+00,0.000000e+00,9.881313e-322,3.397916e-293,...,0.527666,0.791499,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe3+, O2-)"
4,mp-756497,LiFeO2,-6.078770,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[3.04987747 3.81109335 4.24134412] Li, [3.013...",4.143655,3.678547e-82,1.400721e-69,4.517527e-58,1.234025e-47,...,0.527666,0.791499,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe3+, O2-)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,mp-1101736,LiFe3O4,-6.351436,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-1.48982311 2.19943369 -1.43482144] Li, [-1...",5.018471,2.624011e-228,1.241719e-203,2.209240e-180,1.478050e-158,...,0.791499,0.989373,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)"
156,mp-780216,Li5Fe3O8,-5.304033,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.33531488 -4.11912064 -0.49520662] Li, [-0...",3.757134,6.358424e-144,2.845674e-128,1.529721e-113,9.882739e-100,...,0.395749,0.643093,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0,"(Li0+, Fe0+, O0+)"
157,mp-759394,Li2FeO3,-5.613536,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.244898 0. 4.9203229] Li, [1.244898...",3.651901,1.407637e-221,1.260348e-200,1.010579e-180,7.256715e-162,...,0.351777,0.586295,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0,"(Li0+, Fe0+, O0+)"
158,mp-1314324,LiFe2O3,-6.267500,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.52274007 -2.68996763 3.71993333] Li, [ 3...",4.625861,4.635395e-58,4.953273e-49,8.621947e-41,2.444703e-33,...,0.703554,0.938072,0.0,12.0,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)"


In [18]:
from matminer.featurizers.composition.ion import OxidationStates

os_feat = OxidationStates()
data = os_feat.featurize_dataframe(data,"composition_oxid",ignore_errors=True)
data

OxidationStates:   0%|          | 0/160 [00:00<?, ?it/s]

,material_id,pretty_formula,energy_per_atom,spacegroup,structure,density,xrd_0,xrd_1,xrd_2,xrd_3,...,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber,composition_oxid,minimum oxidation state,maximum oxidation state,range oxidation state,std_dev oxidation state
0,mp-756155,Li6FeO4,-5.117326,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.67125537 1.67346391 3.26141012] Li, [1.670...",2.566594,5.977826e-179,1.379972e-160,3.144998e-143,7.076099e-127,...,229.0,217.0,150.090909,100.429752,229.0,"(Li+, Fe2+, O2-)",-2.0,2.0,4.0,2.050825
1,mp-756856,Li5Fe7O12,-6.183322,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0.74461757 4.34819314 2.48026726] Li, [-0.73...",4.516352,1.186238e-56,2.163227e-45,1.940292e-35,8.559855e-27,...,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)",-2.0,3.0,5.0,2.594255
2,mp-762529,Li2(FeO2)5,-5.740312,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-0.35339594 3.48269285 3.02141686] Li, [ 2...",4.046393,1.552619e-78,1.175554e-66,8.898801e-56,6.734917e-46,...,229.0,217.0,101.352941,105.121107,12.0,"(Li0+, Fe0+, O0+)",0.0,0.0,0.0,0.000000
3,mp-18782,LiFeO2,-6.085931,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0. 0. 4.377058] Li, [2.061247 0....",4.231705,0.000000e+00,0.000000e+00,9.881313e-322,3.397916e-293,...,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe3+, O2-)",-2.0,3.0,5.0,2.683282
4,mp-756497,LiFeO2,-6.078770,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[3.04987747 3.81109335 4.24134412] Li, [3.013...",4.143655,3.678547e-82,1.400721e-69,4.517527e-58,1.234025e-47,...,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe3+, O2-)",-2.0,3.0,5.0,2.683282
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,mp-1101736,LiFe3O4,-6.351436,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-1.48982311 2.19943369 -1.43482144] Li, [-1...",5.018471,2.624011e-228,1.241719e-203,2.209240e-180,1.478050e-158,...,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)",-2.0,3.0,5.0,2.544836
156,mp-780216,Li5Fe3O8,-5.304033,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.33531488 -4.11912064 -0.49520662] Li, [-0...",3.757134,6.358424e-144,2.845674e-128,1.529721e-113,9.882739e-100,...,229.0,217.0,120.500000,108.500000,12.0,"(Li0+, Fe0+, O0+)",0.0,0.0,0.0,0.000000
157,mp-759394,Li2FeO3,-5.613536,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.244898 0. 4.9203229] Li, [1.244898...",3.651901,1.407637e-221,1.260348e-200,1.010579e-180,7.256715e-162,...,229.0,217.0,120.500000,108.500000,12.0,"(Li0+, Fe0+, O0+)",0.0,0.0,0.0,0.000000
158,mp-1314324,LiFe2O3,-6.267500,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.52274007 -2.68996763 3.71993333] Li, [ 3...",4.625861,4.635395e-58,4.953273e-49,8.621947e-41,2.444703e-33,...,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)",-2.0,3.0,5.0,2.549510


In [19]:
data = data.drop(["density"],axis=1)
data

,material_id,pretty_formula,energy_per_atom,spacegroup,structure,xrd_0,xrd_1,xrd_2,xrd_3,xrd_4,...,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber,composition_oxid,minimum oxidation state,maximum oxidation state,range oxidation state,std_dev oxidation state
0,mp-756155,Li6FeO4,-5.117326,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.67125537 1.67346391 3.26141012] Li, [1.670...",5.977826e-179,1.379972e-160,3.144998e-143,7.076099e-127,1.571777e-111,...,229.0,217.0,150.090909,100.429752,229.0,"(Li+, Fe2+, O2-)",-2.0,2.0,4.0,2.050825
1,mp-756856,Li5Fe7O12,-6.183322,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0.74461757 4.34819314 2.48026726] Li, [-0.73...",1.186238e-56,2.163227e-45,1.940292e-35,8.559855e-27,1.857379e-19,...,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)",-2.0,3.0,5.0,2.594255
2,mp-762529,Li2(FeO2)5,-5.740312,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-0.35339594 3.48269285 3.02141686] Li, [ 2...",1.552619e-78,1.175554e-66,8.898801e-56,6.734917e-46,5.096179e-37,...,229.0,217.0,101.352941,105.121107,12.0,"(Li0+, Fe0+, O0+)",0.0,0.0,0.0,0.000000
3,mp-18782,LiFeO2,-6.085931,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0. 0. 4.377058] Li, [2.061247 0....",0.000000e+00,0.000000e+00,9.881313e-322,3.397916e-293,7.084615e-266,...,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe3+, O2-)",-2.0,3.0,5.0,2.683282
4,mp-756497,LiFeO2,-6.078770,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[3.04987747 3.81109335 4.24134412] Li, [3.013...",3.678547e-82,1.400721e-69,4.517527e-58,1.234025e-47,2.855097e-38,...,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe3+, O2-)",-2.0,3.0,5.0,2.683282
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,mp-1101736,LiFe3O4,-6.351436,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-1.48982311 2.19943369 -1.43482144] Li, [-1...",2.624011e-228,1.241719e-203,2.209240e-180,1.478050e-158,3.719262e-138,...,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)",-2.0,3.0,5.0,2.544836
156,mp-780216,Li5Fe3O8,-5.304033,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.33531488 -4.11912064 -0.49520662] Li, [-0...",6.358424e-144,2.845674e-128,1.529721e-113,9.882739e-100,7.678617e-87,...,229.0,217.0,120.500000,108.500000,12.0,"(Li0+, Fe0+, O0+)",0.0,0.0,0.0,0.000000
157,mp-759394,Li2FeO3,-5.613536,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.244898 0. 4.9203229] Li, [1.244898...",1.407637e-221,1.260348e-200,1.010579e-180,7.256715e-162,4.666737e-144,...,229.0,217.0,120.500000,108.500000,12.0,"(Li0+, Fe0+, O0+)",0.0,0.0,0.0,0.000000
158,mp-1314324,LiFe2O3,-6.267500,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.52274007 -2.68996763 3.71993333] Li, [ 3...",4.635395e-58,4.953273e-49,8.621947e-41,2.444703e-33,1.129157e-26,...,229.0,217.0,120.500000,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)",-2.0,3.0,5.0,2.549510


In [20]:
from matminer.featurizers.structure import DensityFeatures

de_feat = DensityFeatures()
data = de_feat.featurize_dataframe(data,"structure")
data

DensityFeatures:   0%|          | 0/160 [00:00<?, ?it/s]

,material_id,pretty_formula,energy_per_atom,spacegroup,structure,xrd_0,xrd_1,xrd_2,xrd_3,xrd_4,...,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber,composition_oxid,minimum oxidation state,maximum oxidation state,range oxidation state,std_dev oxidation state,density,vpa,packing fraction
0,mp-756155,Li6FeO4,-5.117326,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.67125537 1.67346391 3.26141012] Li, [1.670...",5.977826e-179,1.379972e-160,3.144998e-143,7.076099e-127,1.571777e-111,...,100.429752,229.0,"(Li+, Fe2+, O2-)",-2.0,2.0,4.0,2.050825,2.566594,9.498197,0.877999
1,mp-756856,Li5Fe7O12,-6.183322,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0.74461757 4.34819314 2.48026726] Li, [-0.73...",1.186238e-56,2.163227e-45,1.940292e-35,8.559855e-27,1.857379e-19,...,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)",-2.0,3.0,5.0,2.594255,4.516352,9.461639,0.683311
2,mp-762529,Li2(FeO2)5,-5.740312,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-0.35339594 3.48269285 3.02141686] Li, [ 2...",1.552619e-78,1.175554e-66,8.898801e-56,6.734917e-46,5.096179e-37,...,105.121107,12.0,"(Li0+, Fe0+, O0+)",0.0,0.0,0.0,0.000000,4.046393,10.937728,0.495092
3,mp-18782,LiFeO2,-6.085931,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[0. 0. 4.377058] Li, [2.061247 0....",0.000000e+00,0.000000e+00,9.881313e-322,3.397916e-293,7.084615e-266,...,108.500000,12.0,"(Li+, Fe3+, O2-)",-2.0,3.0,5.0,2.683282,4.231705,9.298489,0.701018
4,mp-756497,LiFeO2,-6.078770,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[3.04987747 3.81109335 4.24134412] Li, [3.013...",3.678547e-82,1.400721e-69,4.517527e-58,1.234025e-47,2.855097e-38,...,108.500000,12.0,"(Li+, Fe3+, O2-)",-2.0,3.0,5.0,2.683282,4.143655,9.496076,0.686432
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,mp-1101736,LiFe3O4,-6.351436,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[-1.48982311 2.19943369 -1.43482144] Li, [-1...",2.624011e-228,1.241719e-203,2.209240e-180,1.478050e-158,3.719262e-138,...,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)",-2.0,3.0,5.0,2.544836,5.018471,9.863432,0.644696
156,mp-780216,Li5Fe3O8,-5.304033,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.33531488 -4.11912064 -0.49520662] Li, [-0...",6.358424e-144,2.845674e-128,1.529721e-113,9.882739e-100,7.678617e-87,...,108.500000,12.0,"(Li0+, Fe0+, O0+)",0.0,0.0,0.0,0.000000,3.757134,9.122119,0.723315
157,mp-759394,Li2FeO3,-5.613536,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[1.244898 0. 4.9203229] Li, [1.244898...",1.407637e-221,1.260348e-200,1.010579e-180,7.256715e-162,4.666737e-144,...,108.500000,12.0,"(Li0+, Fe0+, O0+)",0.0,0.0,0.0,0.000000,3.651901,8.921715,0.742542
158,mp-1314324,LiFe2O3,-6.267500,"{'symprec': 0.1, 'source': 'spglib', 'symbol':...","[[ 1.52274007 -2.68996763 3.71993333] Li, [ 3...",4.635395e-58,4.953273e-49,8.621947e-41,2.444703e-33,1.129157e-26,...,108.500000,12.0,"(Li+, Fe2+, Fe3+, O2-)",-2.0,3.0,5.0,2.549510,4.625861,9.969109,0.643195


In [21]:
from matminer.datasets import load_dataset

df = load_dataset("matbench_log_kvrh")
df

,structure,log10(K_VRH)
0,"[[0. 0. 0.] Ca, [1.37728887 1.57871271 3.73949...",1.707570
1,"[[3.14048493 1.09300401 1.64101398] Mg, [0.625...",1.633468
2,"[[ 2.06884519 2.40627241 -0.45891585] Si, [1....",1.908485
3,"[[2.06428082 0. 2.06428082] Pd, [0. ...",2.117271
4,"[[3.09635262 1.0689416 1.53602403] Mg, [0.593...",1.690196
...,...,...
10982,"[[0. 0. 0.] Rh, [3.2029368 3.2029368 2.09459...",1.778151
10983,"[[-1.51157821 4.4173925 1.21553922] Mg, [3....",1.724276
10984,"[[4.37546772 4.51128393 6.81784473] H, [0.4573...",1.342423
10985,"[[0. 0. 0.] Si, [ 4.55195829 4.55195829 -4.55...",1.770852


In [ ]:
df_feat = DensityFeatures()
df = df_feat.featurize_dataframe(df,"structure")
df

DensityFeatures:   0%|          | 0/10987 [00:00<?, ?it/s]

In [ ]:
y = df['log10(K_VRH)'].values
y

In [ ]:
excluded = ["structure","log10(K_VRH)"]
features = df.drop(excluded,axis=1)
features

# 机器学习

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
lr = LinearRegression
lr.fit(features,y)

In [ ]:
from pymatgen.ext.matproj import MPRester
from pymatgen.apps.borg.hive import VaspToComputedEntryDrone
from pymatgen.apps.borg.queen import BorgQueen
from pymatgen.entries.compatibility import MaterialsProjectCompatibility
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDPlotter

# Assimilate VASP calculations into ComputedEntry object. Let's assume that
# the calculations are for a series of new LixFeyOz phases that we want to
# know the phase stability.
drone = VaspToComputedEntryDrone()
queen = BorgQueen(drone, rootpath=".")
entries = queen.get_data()

# Obtain all existing Li-Fe-O phases using the Materials Project REST API
with MPRester("DxTbWWBsxEID35nq") as m:
    mp_entries = m.get_entries_in_chemsys(["Li", "Fe", "O"])

# Combined entry from calculated run with Materials Project entries
entries.extend(mp_entries)

# Process entries using the MaterialsProjectCompatibility
compat = MaterialsProjectCompatibility()
entries = compat.process_entries(entries)

# Generate and plot Li-Fe-O phase diagram
pd = PhaseDiagram(entries)
plotter = PDPlotter(pd)
plotter.show()